In [1]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

df = pd.read_csv("YoutubeCommentsDataSet.csv")

df.columns = df.columns.str.strip()

df["Comment"] = df["Comment"].fillna("")

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    words = text.split()

    words = [word for word in words
             if word not in stop_words]

    return " ".join(words)

df["clean_text"] = df["Comment"].apply(clean_text)

print(df[["Comment","clean_text"]].head())

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

tfidf_matrix = tfidf.fit_transform(df["clean_text"])

print("TF-IDF Matrix Shape:")
print(tfidf_matrix.shape)

from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity Matrix Shape:")
print(similarity_matrix.shape)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                             Comment  \
0  lets not forget that apple pay in 2014 require...   
1  here in nz 50 of retailers don’t even have con...   
2  i will forever acknowledge this channel with t...   
3  whenever i go to a place that doesn’t take app...   
4  apple pay is so convenient secure and easy to ...   

                                          clean_text  
0  lets forget apple pay required brand new iphon...  
1  nz retailers dont even contactless credit card...  
2  forever acknowledge channel help lessons ideas...  
3  whenever go place doesnt take apple pay doesnt...  
4  apple pay convenient secure easy use used kore...  
TF-IDF Matrix Shape:
(18408, 5000)
Similarity Matrix Shape:
(18408, 18408)


In [2]:
def recommend(comment_text, top_n=5):

    matches = df[
        df["Comment"] == comment_text
    ]

    if len(matches) == 0:
        return ["Comment not found"]

    idx = matches.index[0]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    recommendations = []

    for i, score in similarity_scores:

        recommendations.append(
            df.iloc[i]["Comment"]
        )

    return recommendations

In [3]:
sample_comment = df.iloc[0]["Comment"]

print("Original Comment:\n")
print(sample_comment)

print("\nRecommendations:\n")

for rec in recommend(sample_comment):

    print("-", rec)

Original Comment:

lets not forget that apple pay in 2014 required a brand new iphone in order to use it a significant portion of apples user base wasnt able to use it even if they wanted to as each successive iphone incorporated the technology and older iphones were replaced the number of people who could use the technology increased

Recommendations:

- to be honest technology has come so far that i cant imagine what an iphone would look like in 2027 i cant wait for the iphone 14 series though love the content thanks for all the apple information keep up the grind
- wow im surprised i figured apples answer to their failure was to buy the new iphone that magically isnt effected by this hack discovered just before the new iphone reveal 
- there were only 3 things i ever wanted from the iphone 1 aod not yet on the iphone 2 split screen pictureinpicture is close enough from the ios 16 3 high refresh rate introduced in iphone 13 propro max ill definitely get the iphone 14 propro max if th

In [4]:
sample_comment = df.iloc[10]["Comment"]

print(recommend(sample_comment))

['well done duda', 'well done', 'make sure to check your answer sheet with someone next to you during the test', 'im going into 10th grade and i want to make sure that i get a head start on chemistry this made everything so much clearer and better', 'very interesting well done']


In [5]:
sample_comment = df.iloc[25]["Comment"]

print(recommend(sample_comment))

['dan the man saving the day riley needs to give him his own theme like brian the electrician', 'thank you brian tracy ', 'alhamdullilahsemoga najib tabah dan terima ini sebangai pengajaran yg amat berharga dlm hidupnya', 'peluang membela diri sejak minggu lalu disiasiakan dengan tindakan bersumpah laknat berpakat malukan cj dan sandiwara memang tiada niat nak bela diri kerana terjerat habis dengan penipuan sendiri', 'klu saham dan kripto crash atau harganya jatuhsaatnya beli pakai uang dingin kemudian sabar dan tungguitu saja']
